In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## What is persistent
- dataset manifests
- split manifests
- clip metadata
- MFCC feature cache
- sklearn-ready flattened arrays
- fitted scalers
- trained models
- evaluation JSON files


In [ ]:
# Optional: install missing dependencies
# Uncomment if needed in a fresh Colab runtime
%pip -q install librosa soundfile scikit-learn joblib tqdm kagglehub

In [ ]:
# Optional: mount Google Drive in Colab for persistence across runtime sessions
# Run this in Colab. If you are local, skip this cell and set PROJECT_ROOT below.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Skipping Drive mount:", e)

Mounted at /content/drive


## Config

Set `PROJECT_ROOT` to a persistent location.
- Colab + Drive : `/content/drive/MyDrive/pf2_speech_baseline`

In [ ]:
import os
import json
import math
import time
import hashlib
import random
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

@dataclass
class Config:
    # persistence root
    PROJECT_ROOT: str = "/content/drive/MyDrive/pf2_speech_baseline"

    # dataset path where the pre-split for-norm dataset should live persistently
    DATA_ROOT: str = "/content/drive/MyDrive/pf2_speech_baseline/data"

    # if DATA_ROOT is missing on first run, download from Kaggle and copy it there
    AUTO_DOWNLOAD_WITH_KAGGLEHUB: bool = True
    KAGGLE_DATASET_ID: str = "mohammedabdeldayem/the-fake-or-real-dataset"

    # audio + MFCC
    SAMPLE_RATE: int = 16000
    CLIP_SECONDS: float = 4.0
    N_MFCC: int = 40
    N_FFT: int = 1024
    HOP_LENGTH: int = 256
    WIN_LENGTH: int = 1024
    FMIN: int = 20
    FMAX: int = 7600

    # feature representation
    # "flatten" is good for LR / SVM
    # "stats" gives mean+std over time and is much smaller
    FEATURE_VIEW: str = "flatten"

    # logistic regression baseline
    LR_C: float = 1.0
    LR_MAX_ITER: int = 2000

cfg = Config()

PROJECT_ROOT = Path(cfg.PROJECT_ROOT)
DATA_ROOT = Path(cfg.DATA_ROOT)

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MANIFEST_DIR = ARTIFACTS_DIR / "manifests"
META_DIR = ARTIFACTS_DIR / "metadata"
FEATURE_DIR = ARTIFACTS_DIR / "features"
ARRAY_DIR = ARTIFACTS_DIR / "arrays"
MODEL_DIR = ARTIFACTS_DIR / "models"
METRIC_DIR = ARTIFACTS_DIR / "metrics"

for d in [PROJECT_ROOT, ARTIFACTS_DIR, MANIFEST_DIR, META_DIR, FEATURE_DIR, ARRAY_DIR, MODEL_DIR, METRIC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MAX_LEN_SAMPLES = int(cfg.SAMPLE_RATE * cfg.CLIP_SECONDS)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)
print("MAX_LEN_SAMPLES =", MAX_LEN_SAMPLES)

PROJECT_ROOT = /content/drive/MyDrive/pf2_speech_baseline
DATA_ROOT    = /content/drive/MyDrive/pf2_speech_baseline/data
MAX_LEN_SAMPLES = 64000


## Dataset acquisition

This is only used once. After the dataset is placed in a persistent folder, future runtime sessions reuse it.

In [ ]:
if not DATA_ROOT.exists() and cfg.AUTO_DOWNLOAD_WITH_KAGGLEHUB:
    import kagglehub
    download_root = Path(kagglehub.dataset_download(cfg.KAGGLE_DATASET_ID))
    print("Downloaded to:", download_root)

    # Try common location for the for-norm subset
    candidate = download_root / "for-norm"
    if candidate.exists():
        DATA_ROOT = candidate
        print("Using:", DATA_ROOT)
    else:
        print("Please inspect downloaded files and set cfg.DATA_ROOT manually.")
else:
    print("Skipping download. Existing DATA_ROOT:", DATA_ROOT.exists())

Skipping download. Existing DATA_ROOT: True


In [ ]:
import os
from pathlib import Path

download_root = Path("/root/.cache/kagglehub/datasets/mohammedabdeldayem/the-fake-or-real-dataset")

for_norm_path = None

for dirpath, _, _ in os.walk(download_root):
    p = Path(dirpath)

    # must contain 'for-norm' in path
    if "for-norm" in str(p).lower():
        if all((p / x).exists() for x in ["training", "validation", "testing"]):
            for_norm_path = p
            print("FOUND for-norm:", for_norm_path)
            break

if for_norm_path is None:
    raise FileNotFoundError("Could not find for-norm dataset inside download")

FileNotFoundError: Could not find for-norm dataset inside download

In [ ]:
!cp -r "/root/.cache/kagglehub/datasets/mohammedabdeldayem/the-fake-or-real-dataset/versions/2/for-norm/for-norm" "/content/drive/MyDrive/pf2_speech_baseline/data/"

cp: cannot stat '/root/.cache/kagglehub/datasets/mohammedabdeldayem/the-fake-or-real-dataset/versions/2/for-norm/for-norm': No such file or directory


## Dataset setup

In [ ]:
import shutil

def dataset_has_expected_structure(root: Path) -> bool:
    expected_dirs = [
        root / "training" / "real",
        root / "training" / "fake",
        root / "validation" / "real",
        root / "validation" / "fake",
        root / "testing" / "real",
        root / "testing" / "fake",
    ]
    return all(p.exists() for p in expected_dirs)

def find_for_norm_root(download_root: Path) -> Path:
    if dataset_has_expected_structure(download_root):
        return download_root

    for dirpath, _, _ in os.walk(download_root):
        candidate = Path(dirpath)
        if dataset_has_expected_structure(candidate):
            return candidate

    raise FileNotFoundError(f"Could not find pre-split for-norm folder inside: {download_root}")

if dataset_has_expected_structure(DATA_ROOT):
    print(f"Using existing dataset at: {DATA_ROOT}")
else:
    if not cfg.AUTO_DOWNLOAD_WITH_KAGGLEHUB:
        raise FileNotFoundError(f"Dataset missing at {DATA_ROOT} and auto-download is disabled.")

    print("Dataset not found. Downloading from Kaggle...")
    import kagglehub

    downloaded_path = Path(kagglehub.dataset_download(cfg.KAGGLE_DATASET_ID))
    print("Downloaded to:", downloaded_path)

    source_root = find_for_norm_root(downloaded_path)
    print("Found dataset root:", source_root)

    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)

    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)

    shutil.copytree(source_root, DATA_ROOT)
    print("Copied dataset to persistent location:", DATA_ROOT)

print("Dataset ready:", DATA_ROOT)

Using existing dataset at: /content/drive/MyDrive/pf2_speech_baseline/data
Dataset ready: /content/drive/MyDrive/pf2_speech_baseline/data


## Dataset inspection

This cell avoids hardcoding `/training/real` unless that structure actually exists.

In [ ]:
AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg", ".m4a"}

def quick_tree(root: Path, depth: int = 2, max_children: int = 8):
    root = Path(root)
    if not root.exists():
        print("Path does not exist:", root)
        return
    print(f"Inspecting: {root}")
    for p in sorted(root.rglob("*")):
        rel = p.relative_to(root)
        level = len(rel.parts)
        if level > depth:
            continue
        indent = "  " * (level - 1)
        print(f"{indent}- {rel}{'/' if p.is_dir() else ''}")

quick_tree(DATA_ROOT, depth=2)

Inspecting: /content/drive/MyDrive/pf2_speech_baseline/data
- for-norm/
  - for-norm/validation/
- testing/
  - testing/fake/
  - testing/real/
- training/
  - training/fake/
  - training/real/
- validation/
  - validation/fake/
  - validation/real/


## Build one reusable raw manifest

This is heavy but persistent stage.  
It scans the dataset once and saves a CSV with:
- filepath
- label
- source split if one already exists in the folders
- file size

The scanner supports:
- `for-norm/training/real/...`, `validation/...`, `testing/...`

In [ ]:
def infer_label_from_parts(parts):
    lowered = [p.lower() for p in parts]
    if "real" in lowered:
        return 0
    if "fake" in lowered:
        return 1
    return None

def infer_existing_split_from_parts(parts):
    lowered = [p.lower() for p in parts]

    if len(lowered) == 0:
        return None

    top = lowered[0]

    if top in ["train", "training"]:
        return "train"
    if top in ["val", "valid", "validation", "dev"]:
        return "val"
    if top in ["test", "testing"]:
        return "test"

    return None

raw_manifest_path = MANIFEST_DIR / "raw_manifest.csv"

def build_raw_manifest(data_root: Path) -> pd.DataFrame:
    rows = []
    for fp in sorted(data_root.rglob("*")):
        if not fp.is_file() or fp.suffix.lower() not in AUDIO_EXTS:
            continue
        parts = fp.relative_to(data_root).parts
        label = infer_label_from_parts(parts)
        if label is None:
            continue
        rows.append({
            "filepath": str(fp),
            "label": label,  # 0=real, 1=fake
            "existing_split": infer_existing_split_from_parts(parts),
            "filesize_bytes": fp.stat().st_size
        })
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(f"No labeled audio files found under {data_root}")
    return df

if raw_manifest_path.exists():
    raw_df = pd.read_csv(raw_manifest_path)
    print("Loaded existing raw manifest:", raw_manifest_path)
else:
    raw_df = build_raw_manifest(DATA_ROOT)
    raw_df.to_csv(raw_manifest_path, index=False)
    print("Built and saved raw manifest:", raw_manifest_path)

print(raw_df.head())
print("\nClass counts:")
print(raw_df["label"].value_counts().sort_index().rename(index={0: "real", 1: "fake"}))
print("\nExisting split counts:")
print(raw_df["existing_split"].fillna("none").value_counts())

Loaded existing raw manifest: /content/drive/MyDrive/pf2_speech_baseline/artifacts/manifests/raw_manifest.csv
                                            filepath  label existing_split  \
0  /content/drive/MyDrive/pf2_speech_baseline/dat...      1            NaN   
1  /content/drive/MyDrive/pf2_speech_baseline/dat...      1            NaN   
2  /content/drive/MyDrive/pf2_speech_baseline/dat...      1            NaN   
3  /content/drive/MyDrive/pf2_speech_baseline/dat...      1            NaN   
4  /content/drive/MyDrive/pf2_speech_baseline/dat...      1            NaN   

   filesize_bytes  
0           56264  
1           85126  
2           56438  
3           36640  
4           39536  

Class counts:
label
real    34605
fake    34919
Name: count, dtype: int64

Existing split counts:
existing_split
train    53868
val      10798
test      4634
none       224
Name: count, dtype: int64


## 5) Build persistent train / val / test manifests

Rules:
- If the dataset already contains train/val/test folders, reuse them directly.
- Otherwise create a stratified split once and save it.

In [ ]:
train_manifest_path = MANIFEST_DIR / "train_manifest.csv"
val_manifest_path = MANIFEST_DIR / "val_manifest.csv"
test_manifest_path = MANIFEST_DIR / "test_manifest.csv"

def make_or_load_split_manifests(raw_df: pd.DataFrame):
    if train_manifest_path.exists() and val_manifest_path.exists() and test_manifest_path.exists():
        train_df = pd.read_csv(train_manifest_path)
        val_df = pd.read_csv(val_manifest_path)
        test_df = pd.read_csv(test_manifest_path)
        print("Loaded existing split manifests.")
        return train_df, val_df, test_df

    # ONLY use existing folder structure
    train_df = raw_df[raw_df["existing_split"] == "train"].copy()
    val_df = raw_df[raw_df["existing_split"] == "val"].copy()
    test_df = raw_df[raw_df["existing_split"] == "test"].copy()

    if train_df.empty or val_df.empty or test_df.empty:
        raise ValueError(
            "Expected pre-split folders (training/validation/testing) not detected correctly."
        )

    train_df.to_csv(train_manifest_path, index=False)
    val_df.to_csv(val_manifest_path, index=False)
    test_df.to_csv(test_manifest_path, index=False)

    print("Saved split manifests from existing dataset folders.")
    return train_df, val_df, test_df

train_df, val_df, test_df = make_or_load_split_manifests(raw_df)

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = df["label"].value_counts().sort_index().to_dict()
    print(f"{name}: n={len(df)}, counts={{real:{counts.get(0,0)}, fake:{counts.get(1,0)}}}")

Loaded existing split manifests.
train: n=53868, counts={real:26941, fake:26927}
val: n=10798, counts={real:5400, fake:5398}
test: n=4634, counts={real:2264, fake:2370}


## 7) Feature-set identity

Every MFCC configuration gets its own cache directory.  
That means you can change `N_MFCC`, `CLIP_SECONDS`, `HOP_LENGTH`, etc. and the notebook will build a **new feature cache** without overwriting the old one.

In [ ]:
import time
import json
import hashlib
from pathlib import Path

feature_config = {
    "sample_rate": cfg.SAMPLE_RATE,
    "clip_seconds": cfg.CLIP_SECONDS,
    "n_mfcc": cfg.N_MFCC,
    "n_fft": cfg.N_FFT,
    "hop_length": cfg.HOP_LENGTH,
    "win_length": cfg.WIN_LENGTH,
    "fmin": cfg.FMIN,
    "fmax": cfg.FMAX,
}

feature_key = hashlib.md5(json.dumps(feature_config, sort_keys=True).encode()).hexdigest()[:12]
feature_set_dir = FEATURE_DIR / f"mfcc_{feature_key}"
feature_set_dir.mkdir(parents=True, exist_ok=True)

feature_config_path = feature_set_dir / "feature_config.json"

for attempt in range(5):
    try:
        with open(feature_config_path, "w") as f:
            json.dump(feature_config, f, indent=2)
        print("Saved feature config:", feature_config_path)
        break

    except OSError as e:
        print(f"Drive write failed on attempt {attempt + 1}/5:", repr(e))
        time.sleep(5)

else:
    print("Could not write feature_config.json to Drive.")
    print("Continuing anyway because this file is only metadata.")

print("Feature cache:", feature_set_dir)
print(json.dumps(feature_config, indent=2))

Drive write failed on attempt 1/5: OSError(5, 'Input/output error')
Drive write failed on attempt 2/5: OSError(5, 'Input/output error')
Saved feature config: /content/drive/MyDrive/pf2_speech_baseline/artifacts/features/mfcc_4db0f011b9c0/feature_config.json
Feature cache: /content/drive/MyDrive/pf2_speech_baseline/artifacts/features/mfcc_4db0f011b9c0
{
  "sample_rate": 16000,
  "clip_seconds": 4.0,
  "n_mfcc": 40,
  "n_fft": 1024,
  "hop_length": 256,
  "win_length": 1024,
  "fmin": 20,
  "fmax": 7600
}


## 8) Persistent MFCC extraction

This is the expensive stage you do not want to repeat.

Strategy:
- one `.npy` file per audio clip
- filename derived from the absolute filepath hash
- reruns only compute missing features

In [ ]:
!pip install tqdm
# import librosa
# from tqdm import tqdm

# def file_hash(path: str) -> str:
#     return hashlib.md5(path.encode()).hexdigest()

# def feature_path_for_audio(path: str) -> Path:
#     return feature_set_dir / f"{file_hash(path)}.npy"

# def load_audio_fixed_length(filepath, sr, max_len):
#     y, _ = librosa.load(filepath, sr=sr, mono=True)
#     if len(y) < max_len:
#         y = np.pad(y, (0, max_len - len(y)))
#     else:
#         y = y[:max_len]
#     return y.astype(np.float32)

# def compute_mfcc(filepath: str) -> np.ndarray:
#     y = load_audio_fixed_length(filepath, cfg.SAMPLE_RATE, MAX_LEN_SAMPLES)
#     mfcc = librosa.feature.mfcc(
#         y=y,
#         sr=cfg.SAMPLE_RATE,
#         n_mfcc=cfg.N_MFCC,
#         n_fft=cfg.N_FFT,
#         hop_length=cfg.HOP_LENGTH,
#         win_length=cfg.WIN_LENGTH,
#         fmin=cfg.FMIN,
#         fmax=cfg.FMAX,
#     )
#     return mfcc.astype(np.float32)

# def materialize_feature_cache(manifest_df: pd.DataFrame, split_name: str):
#     missing = []
#     for fp in manifest_df["filepath"]:
#         out = feature_path_for_audio(fp)
#         if not out.exists():
#             missing.append(fp)

#     print(f"{split_name}: cached={len(manifest_df) - len(missing)}, missing={len(missing)}")
#     for fp in tqdm(missing, desc=f"Caching MFCCs for {split_name}"):
#         mfcc = compute_mfcc(fp)
#         np.save(feature_path_for_audio(fp), mfcc)

# for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
#     materialize_feature_cache(df, split_name)

In [ ]:
import librosa
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import hashlib
from pathlib import Path

def file_hash(path: str) -> str:
    return hashlib.md5(path.encode()).hexdigest()

def feature_path_for_audio(path: str) -> Path:
    return feature_set_dir / f"{file_hash(path)}.npy"

def load_audio_fixed_length(filepath, sr, max_len):
    y, _ = librosa.load(filepath, sr=sr, mono=True)

    if len(y) < max_len:
        y = np.pad(y, (0, max_len - len(y)))
    else:
        y = y[:max_len]

    return y.astype(np.float32)

def compute_mfcc(filepath: str) -> np.ndarray:
    y = load_audio_fixed_length(filepath, cfg.SAMPLE_RATE, MAX_LEN_SAMPLES)

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=cfg.SAMPLE_RATE,
        n_mfcc=cfg.N_MFCC,
        n_fft=cfg.N_FFT,
        hop_length=cfg.HOP_LENGTH,
        win_length=cfg.WIN_LENGTH,
        fmin=cfg.FMIN,
        fmax=cfg.FMAX,
    )

    return mfcc.astype(np.float32)

def materialize_feature_cache(manifest_df: pd.DataFrame, split_name: str):
    bad_rows = []

    missing = []
    for fp in manifest_df["filepath"]:
        out = feature_path_for_audio(fp)
        if not out.exists():
            missing.append(fp)

    print(f"{split_name}: cached={len(manifest_df) - len(missing)}, missing={len(missing)}")

    for fp in tqdm(missing, desc=f"Caching MFCCs for {split_name}"):
        try:
            mfcc = compute_mfcc(fp)
            np.save(feature_path_for_audio(fp), mfcc)

        except Exception as e:
            bad_rows.append({
                "split": split_name,
                "filepath": fp,
                "error": repr(e),
            })
            continue

    if bad_rows:
        bad_df = pd.DataFrame(bad_rows)
        bad_log_path = feature_set_dir / f"bad_audio_{split_name}.csv"
        bad_df.to_csv(bad_log_path, index=False)
        print(f"{split_name}: skipped {len(bad_rows)} bad files")
        print(f"Bad audio log saved to: {bad_log_path}")

for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    materialize_feature_cache(df, split_name)

KeyboardInterrupt: 

In [ ]:
def keep_only_cached_features(df):
    return df[df["filepath"].apply(lambda fp: feature_path_for_audio(fp).exists())].reset_index(drop=True)

train_df = keep_only_cached_features(train_df)
val_df = keep_only_cached_features(val_df)
test_df = keep_only_cached_features(test_df)

print("After removing bad/missing audio:")
print("train:", len(train_df))
print("val:", len(val_df))
print("test:", len(test_df))

After removing bad/missing audio:
train: 53866
val: 10798
test: 4634


## 9) Build reusable model arrays

This stage converts cached MFCC files into arrays that can be reused by many models.

Two views are supported:
- `flatten`: full MFCC image flattened to a vector
- `stats`: mean and std over time for a much smaller feature vector

In [ ]:
def mfcc_to_vector(mfcc: np.ndarray, view: str) -> np.ndarray:
    if view == "flatten":
        return mfcc.reshape(-1).astype(np.float32)
    if view == "stats":
        mean = mfcc.mean(axis=1)
        std = mfcc.std(axis=1)
        return np.concatenate([mean, std]).astype(np.float32)
    raise ValueError(f"Unknown FEATURE_VIEW: {view}")

array_dir = ARRAY_DIR / f"mfcc_{feature_key}_{cfg.FEATURE_VIEW}"
array_dir.mkdir(parents=True, exist_ok=True)

def array_paths(split_name: str):
    return (
        array_dir / f"X_{split_name}.npy",
        array_dir / f"y_{split_name}.npy",
    )

def build_arrays_from_manifest(manifest_df: pd.DataFrame, split_name: str):
    X_path, y_path = array_paths(split_name)
    if X_path.exists() and y_path.exists():
        X = np.load(X_path, mmap_mode="r")
        y = np.load(y_path, mmap_mode="r")
        print(f"Loaded existing arrays for {split_name}: {X.shape}, {y.shape}")
        return X, y

    X_rows = []
    y_rows = []
    for _, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc=f"Building {split_name} arrays"):
        mfcc = np.load(feature_path_for_audio(row["filepath"]))
        vec = mfcc_to_vector(mfcc, cfg.FEATURE_VIEW)
        X_rows.append(vec)
        y_rows.append(int(row["label"]))

    X = np.stack(X_rows).astype(np.float32)
    y = np.array(y_rows, dtype=np.int64)

    np.save(X_path, X)
    np.save(y_path, y)
    print(f"Saved arrays for {split_name}: {X.shape}, {y.shape}")
    return X, y

X_train, y_train = build_arrays_from_manifest(train_df, "train")
X_val, y_val = build_arrays_from_manifest(val_df, "val")
X_test, y_test = build_arrays_from_manifest(test_df, "test")

Building train arrays:   0%|          | 0/53866 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 10) Train-only normalization, saved once

The scaler is fitted only on the training split and reused later for all classical models that need it.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler_path = MODEL_DIR / f"scaler_{feature_key}_{cfg.FEATURE_VIEW}.joblib"

if scaler_path.exists():
    scaler = joblib.load(scaler_path)
    print("Loaded existing scaler:", scaler_path)
else:
    scaler = StandardScaler()
    scaler.fit(np.asarray(X_train))
    joblib.dump(scaler, scaler_path)
    print("Saved scaler:", scaler_path)

X_train_scaled = scaler.transform(np.asarray(X_train))
X_val_scaled = scaler.transform(np.asarray(X_val))
X_test_scaled = scaler.transform(np.asarray(X_test))
print(X_train_scaled.shape, X_val_scaled.shape, X_test_scaled.shape)

## 11) Metrics helper

In [ ]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

def compute_metrics(y_true, y_pred, y_score):
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_score)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr_human_as_fake = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    return {
        "accuracy": float(acc),
        "precision_fake": float(precision),
        "recall_fake": float(recall),
        "f1_fake": float(f1),
        "roc_auc": float(roc_auc),
        "false_positive_rate_human_as_fake": float(fpr_human_as_fake),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

def save_json(obj, path: Path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

## 12) Logistic Regression baseline

This stage is also persistent:
- if the model and metrics already exist, load them
- otherwise train and save them

You can clone this pattern for SVM, Random Forest, XGBoost, or a PyTorch model while reusing the same manifests and feature cache.

In [ ]:

from sklearn.linear_model import LogisticRegression

model_name = f"logreg_C{cfg.LR_C}_{feature_key}_{cfg.FEATURE_VIEW}"
model_path = MODEL_DIR / f"{model_name}.joblib"
val_metric_path = METRIC_DIR / f"{model_name}_val.json"
test_metric_path = METRIC_DIR / f"{model_name}_test.json"

if model_path.exists() and val_metric_path.exists() and test_metric_path.exists():
    model = joblib.load(model_path)
    val_metrics = json.load(open(val_metric_path))
    test_metrics = json.load(open(test_metric_path))
    print("Loaded existing model and metrics.")
else:
    model = LogisticRegression(
        C=cfg.LR_C,
        max_iter=cfg.LR_MAX_ITER,
        solver="saga",
        random_state=SEED,
        n_jobs=-1
    )
    model.fit(X_train_scaled, y_train)

    val_prob = model.predict_proba(X_val_scaled)[:, 1]
    val_pred = (val_prob >= 0.5).astype(int)
    val_metrics = compute_metrics(y_val, val_pred, val_prob)

    test_prob = model.predict_proba(X_test_scaled)[:, 1]
    test_pred = (test_prob >= 0.5).astype(int)
    test_metrics = compute_metrics(y_test, test_pred, test_prob)

    joblib.dump(model, model_path)
    save_json(val_metrics, val_metric_path)
    save_json(test_metrics, test_metric_path)
    print("Trained and saved model + metrics.")

print("Validation metrics")
print(json.dumps(val_metrics, indent=2))
print("\nTest metrics")
print(json.dumps(test_metrics, indent=2))

## Pytorch Models

### Shared PyTorch setup

In [ ]:
%pip -q install torch

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
import numpy as np
import json
import time

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

N_MFCC = cfg.N_MFCC
N_FRAMES = X_train_scaled.shape[1] // N_MFCC
print("MFCC shape per sample:", N_MFCC, "x", N_FRAMES)

def to_mfcc_tensor(X):
    X = np.asarray(X, dtype=np.float32)
    X = X.reshape(len(X), 1, N_MFCC, N_FRAMES)
    return torch.tensor(X, dtype=torch.float32)

Xtr_t = to_mfcc_tensor(X_train_scaled)
Xva_t = to_mfcc_tensor(X_val_scaled)
Xte_t = to_mfcc_tensor(X_test_scaled)

ytr_t = torch.tensor(np.asarray(y_train), dtype=torch.float32)
yva_t = torch.tensor(np.asarray(y_val), dtype=torch.float32)
yte_t = torch.tensor(np.asarray(y_test), dtype=torch.float32)

BATCH_SIZE = 128

train_loader = DataLoader(
    TensorDataset(Xtr_t, ytr_t),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    TensorDataset(Xva_t, yva_t),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    TensorDataset(Xte_t, yte_t),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

pos_count = float(np.sum(y_train == 1))
neg_count = float(np.sum(y_train == 0))
pos_weight = torch.tensor([neg_count / max(pos_count, 1.0)], device=DEVICE)
print("pos_weight:", pos_weight.item())

### Shared training/evaluation function

In [ ]:
def predict_torch(model, loader):
    model.eval()
    probs = []
    labels = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb).view(-1)
            prob = torch.sigmoid(logits).detach().cpu().numpy()

            probs.extend(prob.tolist())
            labels.extend(yb.numpy().tolist())

    probs = np.array(probs)
    labels = np.array(labels).astype(int)
    preds = (probs >= 0.5).astype(int)

    return labels, preds, probs


def train_torch_model(model, model_name, epochs=12, lr=1e-3, patience=3):
    model = model.to(DEVICE)

    model_path = MODEL_DIR / f"{model_name}.pt"
    val_metric_path = METRIC_DIR / f"{model_name}_val.json"
    test_metric_path = METRIC_DIR / f"{model_name}_test.json"

    if model_path.exists() and val_metric_path.exists() and test_metric_path.exists():
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        val_metrics = json.load(open(val_metric_path))
        test_metrics = json.load(open(test_metric_path))
        print("Loaded existing model:", model_name)
        print("Validation metrics")
        print(json.dumps(val_metrics, indent=2))
        print("\nTest metrics")
        print(json.dumps(test_metrics, indent=2))
        return model, val_metrics, test_metrics

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=1
    )

    best_val_f1 = -1
    best_state = None
    bad_epochs = 0

    for epoch in range(1, epochs + 1):
        start = time.time()
        model.train()
        total_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb).view(-1)
            loss = criterion(logits, yb)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)

            optimizer.step()
            total_loss += loss.item() * xb.size(0)

        train_loss = total_loss / len(train_loader.dataset)

        yv, pv, sv = predict_torch(model, val_loader)
        val_metrics = compute_metrics(yv, pv, sv)
        scheduler.step(val_metrics["f1_fake"])

        print(
            f"Epoch {epoch:02d} | "
            f"loss={train_loss:.4f} | "
            f"val_f1={val_metrics['f1_fake']:.4f} | "
            f"val_auc={val_metrics['roc_auc']:.4f} | "
            f"time={time.time() - start:.1f}s"
        )

        if val_metrics["f1_fake"] > best_val_f1:
            best_val_f1 = val_metrics["f1_fake"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            print("Early stopping.")
            break

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), model_path)

    yv, pv, sv = predict_torch(model, val_loader)
    yt, pt, st = predict_torch(model, test_loader)

    val_metrics = compute_metrics(yv, pv, sv)
    test_metrics = compute_metrics(yt, pt, st)

    save_json(val_metrics, val_metric_path)
    save_json(test_metrics, test_metric_path)

    print("\nSaved:", model_path)
    print("\nValidation metrics")
    print(json.dumps(val_metrics, indent=2))
    print("\nTest metrics")
    print(json.dumps(test_metrics, indent=2))

    return model, val_metrics, test_metrics

### Model 1: CNN + BiLSTM + Attention

In [ ]:
class CNNBiLSTMAttention(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 1)),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 1)),
        )

        self.lstm = nn.LSTM(
            input_size=64 * (N_MFCC // 4),
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.attn = nn.Sequential(
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.35),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.cnn(x)                 # [B, C, MFCC, T]
        x = x.permute(0, 3, 1, 2)       # [B, T, C, MFCC]
        x = x.flatten(start_dim=2)      # [B, T, features]

        h, _ = self.lstm(x)             # [B, T, 128]

        weights = torch.softmax(self.attn(h), dim=1)
        context = torch.sum(weights * h, dim=1)

        return self.classifier(context).squeeze(1)


model_name = f"cnn_bilstm_attention_{feature_key}_{cfg.FEATURE_VIEW}"
cnn_bilstm_attn, val_metrics, test_metrics = train_torch_model(
    CNNBiLSTMAttention(),
    model_name,
    epochs=12,
    lr=1e-3,
    patience=3
)

### Model 2: Temporal Convolutional Network

In [ ]:
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation

        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return self.net(x) + self.downsample(x)


class TCNClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.tcn = nn.Sequential(
            TemporalBlock(N_MFCC, 64, kernel_size=3, dilation=1),
            TemporalBlock(64, 64, kernel_size=3, dilation=2),
            TemporalBlock(64, 128, kernel_size=3, dilation=4),
            TemporalBlock(128, 128, kernel_size=3, dilation=8),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x.squeeze(1)        # [B, MFCC, T]
        x = self.tcn(x)
        return self.classifier(x).squeeze(1)


model_name = f"tcn_{feature_key}_{cfg.FEATURE_VIEW}"
tcn_model, val_metrics, test_metrics = train_torch_model(
    TCNClassifier(),
    model_name,
    epochs=12,
    lr=1e-3,
    patience=3
)

### Model 3: CNN + Transformer Encoder

In [ ]:
class CNNTransformerEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 1)),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 1)),
        )

        feature_dim = 64 * (N_MFCC // 4)

        self.proj = nn.Linear(feature_dim, 128)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=4,
            dim_feedforward=256,
            dropout=0.2,
            batch_first=True,
            activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.classifier = nn.Sequential(
            nn.LayerNorm(128),
            nn.Dropout(0.35),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.cnn(x)                 # [B, C, MFCC, T]
        x = x.permute(0, 3, 1, 2)       # [B, T, C, MFCC]
        x = x.flatten(start_dim=2)      # [B, T, features]

        x = self.proj(x)
        x = self.encoder(x)

        x = x.mean(dim=1)
        return self.classifier(x).squeeze(1)


model_name = f"cnn_transformer_{feature_key}_{cfg.FEATURE_VIEW}"
cnn_transformer, val_metrics, test_metrics = train_torch_model(
    CNNTransformerEncoder(),
    model_name,
    epochs=12,
    lr=7e-4,
    patience=3
)

### Model 4: ResNet-style CNN with SE attention

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()

        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        weights = self.net(x).view(x.size(0), x.size(1), 1, 1)
        return x * weights


class SEResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, stride=stride),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),

            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            SEBlock(out_ch)
        )

        self.shortcut = (
            nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride)
            if in_ch != out_ch or stride != 1
            else nn.Identity()
        )

        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.block(x) + self.shortcut(x))


class SEResNetMFCC(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            SEResBlock(32, 32),
            SEResBlock(32, 64, stride=2),
            SEResBlock(64, 128, stride=2),

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),

            nn.Dropout(0.35),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


model_name = f"se_resnet_mfcc_{feature_key}_{cfg.FEATURE_VIEW}"
se_resnet, val_metrics, test_metrics = train_torch_model(
    SEResNetMFCC(),
    model_name,
    epochs=12,
    lr=1e-3,
    patience=3
)

## 13) Model registry summary

This gives you a reusable summary table across multiple saved models.

In [ ]:

metric_files = sorted(METRIC_DIR.glob("*_test.json"))
rows = []
for fp in metric_files:
    metrics = json.load(open(fp))
    metrics["model_file"] = fp.name
    rows.append(metrics)

summary_df = pd.DataFrame(rows)
summary_df = summary_df[[
    "model_file",
    "accuracy",
    "precision_fake",
    "recall_fake",
    "f1_fake",
    "roc_auc",
    "false_positive_rate_human_as_fake",
    "tn", "fp", "fn", "tp"
]] if not summary_df.empty else pd.DataFrame()

summary_df

## 14) How to reuse this notebook for other models

Use the same saved artifacts:
- `artifacts/manifests/*.csv`
- `artifacts/features/mfcc_<feature_key>/`
- `artifacts/arrays/mfcc_<feature_key>_<view>/`
- `artifacts/models/scaler_<...>.joblib`

Then add a new training cell for another model, for example:
- Linear SVM
- RBF SVM
- Random Forest
- MLP
- CNN on MFCC maps

Only the new model-training stage runs. The heavy preprocessing stays cached.

In [ ]:
# Example stub for another model
# from sklearn.svm import LinearSVC
# svm = LinearSVC(C=1.0, random_state=SEED)
# svm.fit(X_train_scaled, y_train)
# Save the model and metrics with the same pattern as above.